In [ ]:
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_with_ground_truth_and_multiple_choices
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS


df = pd.read_csv("./data/_OPC_full_dataset.csv")
df.rename(columns={"problem":"math_problem", "solution":"math_proof"}, inplace=True)
# df = df[df['level'] == "undergraduate"]
df['proof_length'] = df['math_proof'].apply(lambda x: len(x))
df = df[df['proof_length'] >= 4000]
#filter out cp
df_correct = df[(df['score'] == '[1]') | (df['score'] == '[1 1]')] # correct proofs
df_incorrect = df[(df['score'] == '[0]') | (df['score'] == '[0 0]')] # incorrect proofs
df_correct['score'] = True
df_incorrect['score'] = False
df_incorrect

In [ ]:
experiment_name = "math_proofs"
number_of_choices = 5
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_block"]
user_prompt_template_repeated_attribution_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_attribution_block"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block))

In [ ]:
# model_name = "gpt-4o-mini"
# model_name = "gpt-5"
from os import name


model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})

user_prompt_template = build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block)

names_dict = {
"name_1": "P.P.",
"name_2": "A.J.",
"name_3": "L.K.",
"name_4": "M.R.",
"name_5": "S.T.",
}
political_attitudes = {
"political_attitude_1": "conservative",
"political_attitude_2": "progressive",
"political_attitude_3": "left-leaning",
"political_attitude_4": "right-wing",
"political_attitude_5": "Republican",
}

stimuli_factors_into_user_prompt = {
"math_problem_1": df_incorrect.iloc[0]['math_problem'],
"math_proof_1": df_incorrect.iloc[0]['math_proof'],
"math_problem_2": df_incorrect.iloc[0]['math_problem'],
"math_proof_2": df_incorrect.iloc[0]['math_proof'],
"math_problem_3": df_incorrect.iloc[0]['math_problem'],
"math_proof_3": df_incorrect.iloc[0]['math_proof'],
"math_problem_4": df_correct.iloc[0]['math_problem'],
"math_proof_4": df_correct.iloc[0]['math_proof'],
"math_problem_5": df_incorrect.iloc[0]['math_problem'],
"math_proof_5": df_incorrect.iloc[0]['math_proof'],
}

user_prompt = user_prompt_template.format(**names_dict,
                                                      **political_attitudes,
                                                      **stimuli_factors_into_user_prompt)


messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:


models = ["gpt-5-mini"]


n = 50
custom_model_kwargs = {}
stimuli_factors = ["math_problem", "math_proof"]
additional_variables_from_df_to_save = [] 
number_of_choices = 5
path_to_save_model_outputs = "./comparative_experiment_with_ground_truth_and_multiple_choices"
random_seed = 42


In [ ]:
payloads = await carry_out_comparative_experiment_with_ground_truth_and_multiple_choices(models=models, df_correct=df_correct, df_incorrect=df_incorrect, n=n, 
                                                                                         system_prompt=system_prompt, user_prompt_template_repeated_block=user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block=user_prompt_template_repeated_attribution_block,
                                                                                         stimuli_factors=stimuli_factors, additional_variables_from_df_to_save=additional_variables_from_df_to_save,
                                                                                         custom_model_kwargs=custom_model_kwargs, path_to_save_model_outputs=path_to_save_model_outputs,
                                                                                         random_seed=21, number_of_choices=number_of_choices)

df = pd.DataFrame(payloads)
df['model_response_pole'].value_counts()